# 02 - CV Model Training Experiments

This notebook explores CV model training for flood image classification:
- Backbone comparison (MobileNetV3-Small, EfficientNet-B0, ResNet50)
- Quick training run with MobileNetV3-Small
- Evaluation metrics and confusion matrix visualization

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parent))

from cv_classifier.model import FloodClassifier, BACKBONE_REGISTRY, SEVERITY_CLASSES
from cv_classifier.dataset import (
    prepare_dataloaders,
    generate_synthetic_samples,
    SyntheticFloodDataset,
)

import torch
import numpy as np
from collections import Counter

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Backbones: {list(BACKBONE_REGISTRY.keys())}")

In [ ]:
samples = generate_synthetic_samples(n_per_class=60)

class_counts = Counter(s["label"] for s in samples)
print("Class distribution:")
for cls in SEVERITY_CLASSES:
    print(f"  {cls}: {class_counts[cls]}")

depths = [s["depth_cm"] for s in samples]
print(f"\nDepth range: {min(depths):.1f} - {max(depths):.1f} cm")
print(f"Total samples: {len(samples)}")

## Backbone Comparison

In [ ]:
print(f"{'Backbone':<25} {'Params':>12} {'Feature Dim':>12}")
print("-" * 50)

for name, (model_fn, weights_fn, feat_dim) in BACKBONE_REGISTRY.items():
    temp_model = FloodClassifier(num_classes=4, pretrained=False, backbone=name)
    n_params = sum(p.numel() for p in temp_model.parameters())
    print(f"{name:<25} {n_params:>12,} {feat_dim:>12}")
    del temp_model

## Training with MobileNetV3-Small

In [ ]:
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler

from cv_classifier.train import train_one_epoch, validate, compute_class_weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 3
BATCH_SIZE = 16

loaders = prepare_dataloaders(image_size=224, batch_size=BATCH_SIZE, augment=True)

model = FloodClassifier(num_classes=4, pretrained=True, backbone="mobilenet_v3_small").to(device)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

cls_weights = compute_class_weights(loaders["train_samples"]).to(device)
optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = GradScaler(enabled=(device.type == "cuda"))

for epoch in range(1, EPOCHS + 1):
    train_m = train_one_epoch(model, loaders["train"], optimizer, 0.5, 0.5, device, scaler)
    val_m = validate(model, loaders["val"], 0.5, 0.5, device)
    scheduler.step()
    lr = optimizer.param_groups[0]["lr"]
    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train_loss={train_m['loss']:.4f} acc={train_m['accuracy']:.4f} | "
        f"val_loss={val_m['loss']:.4f} acc={val_m['accuracy']:.4f} | "
        f"lr={lr:.6f}"
    )

## Evaluation

In [ ]:
from cv_classifier.evaluate import load_model, evaluate as run_evaluate
import json

eval_results = run_evaluate()

print("\n=== Summary ===")
print(f"Test size: {eval_results['test_size']}")
print(f"Accuracy:  {eval_results['classification']['accuracy']}")
print(f"F1:        {eval_results['classification']['f1_weighted']}")
print(f"AUC-ROC:   {eval_results['classification']['auc_roc']}")
print(f"Depth MAE: {eval_results['regression']['mae_cm']} cm")
print(f"Depth RMSE:{eval_results['regression']['rmse_cm']} cm")
print(f"R2:        {eval_results['regression']['r2']}")

## Confusion Matrix Visualization

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

results_path = Path.cwd().resolve().parent / "cv_classifier" / "checkpoints" / "evaluation_results.json"
with open(results_path) as f:
    results = json.load(f)

cm = np.array(results["confusion_matrix"])

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)

labels = SEVERITY_CLASSES
ax.set(
    xticks=np.arange(len(labels)),
    yticks=np.arange(len(labels)),
    xticklabels=labels,
    yticklabels=labels,
    ylabel="True",
    xlabel="Predicted",
    title="Confusion Matrix - Flood Severity Classification",
)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j, i, format(cm[i, j], "d"),
            ha="center", va="center",
            color="white" if cm[i, j] > thresh else "black",
        )

fig.tight_layout()
plt.show()